In [142]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from xgboost import XGBClassifier
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split


In [143]:
df_train = pd.read_csv("train.csv")
df_test = pd.read_csv("test.csv")

In [144]:
# thông số cơ bản
print(df_train.shape)
display(df_train.head())
print(df_train.info())

(8693, 14)


,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


<class 'pandas.DataFrame'>
RangeIndex: 8693 entries, 0 to 8692
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   PassengerId   8693 non-null   str    
 1   HomePlanet    8492 non-null   str    
 2   CryoSleep     8476 non-null   object 
 3   Cabin         8494 non-null   str    
 4   Destination   8511 non-null   str    
 5   Age           8514 non-null   float64
 6   VIP           8490 non-null   object 
 7   RoomService   8512 non-null   float64
 8   FoodCourt     8510 non-null   float64
 9   ShoppingMall  8485 non-null   float64
 10  Spa           8510 non-null   float64
 11  VRDeck        8505 non-null   float64
 12  Name          8493 non-null   str    
 13  Transported   8693 non-null   bool   
dtypes: bool(1), float64(6), object(2), str(5)
memory usage: 891.5+ KB
None


In [145]:
# kiểm tra missing values
table = pd.DataFrame({
    "count": df_train.isnull().sum(),
    "percentage": df_train.isnull().mean()
})
display(table)

,count,percentage
PassengerId,0,0.000000
HomePlanet,201,0.023122
CryoSleep,217,0.024963
Cabin,199,0.022892
Destination,182,0.020936
Age,179,0.020591
VIP,203,0.023352
RoomService,181,0.020821
FoodCourt,183,0.021051
ShoppingMall,208,0.023927


In [146]:
# kiểm tra phân bố
print(df_train["Transported"].value_counts())

Transported
True     4378
False    4315
Name: count, dtype: int64


In [147]:
# tiền xử lí Cabin
df_train[["Deck","CabinNum","Side"]] = df_train["Cabin"].str.split("/", expand=True)
df_test[["Deck","CabinNum","Side"]] = df_test["Cabin"].str.split("/", expand=True)

In [148]:
# tiền xử lí PassengerId
df_train[["Group","Member"]] = df_train["PassengerId"].str.split("_", expand=True)
df_test[["Group","Member"]] = df_test["PassengerId"].str.split("_", expand=True)

In [149]:
# bỏ cột Name
df_train.drop(columns=["Name"], inplace=True)
df_test.drop(columns=["Name"], inplace=True)

In [150]:
# Xử lí missing values, encoder, chia tập 
# Các cột số
num_features = [
    "Age",
    "RoomService",
    "FoodCourt",
    "ShoppingMall",
    "Spa",
    "VRDeck",
    "CabinNum"
]

# Các cột phân loại
cat_features = [
    "HomePlanet",
    "CryoSleep",
    "Destination",
    "VIP",
    "Deck",
    "Side"
]

# Pipeline cho dữ liệu số
num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

# Pipeline cho dữ liệu phân loại
cat_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

# Kết hợp hai pipeline
preprocessor = ColumnTransformer([
    ("num", num_pipeline, num_features),
    ("cat", cat_pipeline, cat_features)
])
X = df_train.drop(columns="Transported")
y = df_train["Transported"]

X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

X_train = preprocessor.fit_transform(X_train)
X_valid = preprocessor.transform(X_valid)
X_test = preprocessor.transform(df_test)

print(X_train)
print(y_train)

[[  0.   0.   0. ...   0.   0.   1.]
 [ 17.   0.   0. ...   0.   0.   1.]
 [ 35.   0.   0. ...   0.   0.   1.]
 ...
 [ 45.   1.   7. ...   0.   0.   1.]
 [ 24.   0.   0. ...   0.   0.   1.]
 [ 21.  32. 640. ...   0.   0.   1.]]
3600     True
1262     True
8612    False
5075     True
4758    False
        ...  
4087     True
4406    False
7111     True
426      True
7925     True
Name: Transported, Length: 6954, dtype: bool


In [151]:
# tạo các mô hình
rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)

rf_pred = rf.predict(X_valid)

xgb = XGBClassifier(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    random_state=42,
    eval_metric="logloss"
)

xgb.fit(X_train, y_train)
xgb_pred = xgb.predict(X_valid)

estimators = [
    ("rf", rf),
    ("xgb", xgb)
]

stack = StackingClassifier(
    estimators=estimators,
    final_estimator=LogisticRegression(),
    cv=5,
    n_jobs=-1
)

stack.fit(X_train, y_train)
stack_pred = stack.predict(X_valid)

rf_acc = accuracy_score(y_valid, rf_pred)
xgb_acc = accuracy_score(y_valid, xgb_pred)
stack_acc = accuracy_score(y_valid, stack_pred)

In [152]:
# benchmark
benchmark = pd.DataFrame({
    "Model": [
        "Random Forest",
        "XGBoost",
        "Stacking"
    ],
    "Technique": [
        "Bagging",
        "Boosting",
        "Stacking"
    ],
    "Accuracy": [
        rf_acc,
        xgb_acc,
        stack_acc
    ]
})

benchmark = benchmark.sort_values(
    by="Accuracy",
    ascending=False
)

print(benchmark)

           Model Technique  Accuracy
1        XGBoost  Boosting  0.821162
2       Stacking  Stacking  0.820012
0  Random Forest   Bagging  0.810236


In [153]:
# tạo file nộp

test_pred = xgb.predict(X_test)
submission = pd.DataFrame({
    "PassengerId": df_test["PassengerId"],
    "Transported": test_pred.astype(bool)
})

submission.to_csv("submission.csv", index=False)